# LSA Tests
Run LSA retrieval on a small query subset and evaluate against Cranfield relevance judgments.

In [1]:
import os
import sys
import pandas as pd
import nltk

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

project_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = os.path.join(project_dir, "data")
utils_dir = os.path.join(project_dir, "source", "utils")

for path in (project_dir, utils_dir):
    if path not in sys.path:
        sys.path.append(path)

from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex
from source.utils.LSA import LSA

In [4]:
documents_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\documents.csv'
queries_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\cranfield\queries.csv'

documents_df = pd.read_csv(documents_path)
queries_df = pd.read_csv(queries_path)

tp = TextPreprocessing()
processed_docs = tp.process_dataframe(documents_df)
vocab = tp.get_vocab()

index_builder = InvertedIndex(processed_docs, vocab)
index_builder._build()

lsa = LSA(processed_docs, index_builder, vocab, dims=200)
lsa._vectorize(use_tfidf=True)
lsa._fit_svd()

In [5]:
sample_queries = queries_df.head(5)
top_k = 10
doc_map = documents_df.set_index("id")["content"].to_dict()

results = []
for _, row in sample_queries.iterrows():
    query_id = row["id"]
    query_text = row["content"]
    ranked = lsa.search(query_text, top_k=top_k)
    enriched = []
    for doc_id, score in ranked:
        enriched.append({
            "doc_id": doc_id,
            "score": float(score),
            "content": doc_map.get(doc_id, "")
        })
    results.append({"query_id": query_id, "query": query_text, "results": enriched})

results

[{'query_id': 1,
  'query': ' what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft .',
  'results': [{'doc_id': 486,
    'score': 0.592117965221405,
    'content': 'similarity laws for aerothermoelastic testing . the similarity laws for aerothermoelastic testing are presented in the range .  these are obtained by making nondimensional the appropriate governing equations of the individual external aerodynamic flow, heat conduction to the interior, and stress deflection problems which make up the combined aerothermoelastic problem . for the general aerothermoelastic model, where the model is placed in a high stagnation temperature wind tunnel, similitude is shown to be very difficult to achieve for a scale ratio other than unity .  the primary conflict occurs between the free stream mach number reynolds number aeroelastic parameter heat conduction parameter and thermal expansion parameter . means of dealing with this basic conflict are pr

# Evaluation Metrics (REL Ground Truth)
Compute MAP@25, NDCG@25, Precision/Recall/F1 at 25 using Cranfield relevance judgments.

In [7]:
import os

from source.utils.evaluations import Evaluator

rel_dir = os.path.abspath(os.path.join(project_dir, "..", "Dataset", "Cranfield", "REL"))
cranqrel_path = os.path.abspath(os.path.join(project_dir, "..", "Dataset", "cran", "cranqrel"))


def load_relations(rel_folder, fallback_file=None):
    qrels = {}
    if os.path.isdir(rel_folder):
        files = [name for name in os.listdir(rel_folder) if name.endswith(".txt")]
        for name in files:
            path = os.path.join(rel_folder, name)
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 3:
                        continue
                    qid = int(parts[0])
                    doc_id = int(parts[1])
                    rel = float(parts[2])
                    qrels.setdefault(qid, {})[doc_id] = rel
        if qrels:
            return qrels

    if fallback_file and os.path.isfile(fallback_file):
        with open(fallback_file, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                qid = int(parts[0])
                doc_id = int(parts[1])
                rel = float(parts[2])
                qrels.setdefault(qid, {})[doc_id] = rel

    return qrels


qrels = load_relations(rel_dir, cranqrel_path)

k = 10
queries = queries_df.to_dict(orient="records")

lsa_evaluator = Evaluator(
    queries=queries,
    qrels=qrels,
    retriever=lsa,
    normalize_relevance_scores=True,
)

result = lsa_evaluator.evaluate_all(top_k=k, return_results=False, return_df=False)
metrics = result.summary

metrics

{'Precision@10': 0.2408888888888889,
 'Recall@10': 0.4014672778011271,
 'F1@10': 0.27130856864794767,
 'MAP@10': 0.26322603510540016,
 'NDCG@10': 0.3436992738681533}